# 09.2 哼唱检索：从一段哼唱到旋律命中

哼唱检索（query by humming, QBH）接收用户重新唱出或哼出的旋律片段，而不是库内录音的截取。查询的音高和节奏可能不准确，也常从中间乐句开始，还可能多唱或漏掉几个音。

哼唱与原曲在音色、绝对音高和速度上通常不同，因此局部声学峰不再适合作为主要匹配依据。系统一般假定查询仍保留部分旋律走向、相对音程或节奏关系，但保留程度因人和片段而异。

本 Notebook 把查询与库内旋律表示为以半音为单位的一维音高轮廓，再用动态时间规整（dynamic time warping, DTW）进行弹性对齐。排名使用按对齐路径长度归一化的累计代价；这只是 QBH 的一种代表性实现。


## 0. 环境与数据

旋律库分为两层。核心层包含八首 MIDI，包括《茉莉花》《友谊地久天长》等，便于逐首核对结果；规模层从 Essen 民歌集中国子集中选取 414 首，用来观察增加候选后排名和耗时的变化。

查询以合成哼唱为主，每种扰动都可独立调节。最后一节再说明真实哼唱带来的额外误差，并给出公开数据集指针。


In [ ]:
from pathlib import Path
import sys
import time
import warnings

# 路径推断：从 cwd 向上找含 CODE/chapter09/_common 的目录；ROOT 指向 CODE/chapter09/
_p = Path.cwd()
while not (_p / "CODE" / "chapter09" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter09/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter09"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from _common.audio_io import load_audio  # audio_io 先设 NUMBA_CACHE_DIR，须早于 librosa 导入
from _common.env_check import check_notebook_env
from _common.humming import HummingParams, render_humming
from _common.melody import crop_notes, load_midi_notes, transpose_notes
from _common.paths import portable_path
from _common.plotting import LINE_GRAYS, finish_figure, setup_plot_style

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 民歌 MIDI 的 tempo 事件不在第 0 轨，pretty_midi 会警告，读取不受影响
warnings.filterwarnings("ignore", message=r"Tempo, Key or Time signature.*", category=RuntimeWarning)
check_notebook_env("09_2_qbh_dtw")

SR = 22050
PYIN_HOP = 1024                 # 轮廓帧率约 21.5 Hz
FRAME_RATE = SR / PYIN_HOP
PYIN_FMIN = librosa.note_to_hz("C3")
PYIN_FMAX = librosa.note_to_hz("C7")
QUERY_TRANSPOSE = 5             # 哼唱不按原调起音，统一移高 5 半音
QUERY_SEC = 45.0                # 鲁棒性与规模实验的查询片段长度
DEFAULT_DETUNE = 25.0           # 合成哼唱默认扰动：音高偏移标准差(音分)
DEFAULT_WANDER = 0.03           # 节奏漂移强度(秒)

DATASETS = ROOT.parent / "datasets"
MELODY_DIR = DATASETS / "melodies"
ESSEN_DIR = DATASETS / "essen_china_midi"
AUTHOR_HUM_DIR = DATASETS / "audio_author" / "chapter_09_author" / "humming"
OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
for path in [OUTPUT_FIGURES, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)

core_songs = {}
rows = []
for midi_path in sorted(MELODY_DIR.glob("*.midi")):
    notes = load_midi_notes(midi_path)
    core_songs[midi_path.stem] = notes
    rows.append({"旋律": midi_path.stem, "音符数": len(notes), "时长(秒)": round(notes[-1].end, 1)})
print(f"核心旋律库 {len(core_songs)} 首")
print(pd.DataFrame(rows).to_string(index=False))


## 1. 从哼唱到音高轮廓

合成器从 MIDI 音符序列生成波形，并加入四类可控变化：逐音随机音高偏移、整段平滑节奏漂移、相邻音符间的滑音，以及单音内部逐渐增强的颤音。渲染时保存真实音高轨迹，供提取结果对照。

查询端沿用第 5 章的 pYIN，逐帧估计基频和有声标记。基频换算为半音单位后删除无声帧，得到帧级音高轮廓。删除无声帧会压缩停顿，因此后续序列的索引不再完整保留原始时间轴。

匹配前，每条轮廓减去自身中位数，以近似消除整体音高偏移。查询若只是全曲片段，其中位音高可能与完整库曲不同，所以这种中心化并不提供严格的移调不变性。更完整的方案可比较音高差分，或显式扫描全局移调量。

库端直接从 MIDI 音符按约 21.5 Hz 采样为轮廓，同样做中位数中心化，不经过音频基频提取。实际曲库既可来自音频，也可来自 MIDI、乐谱或人工旋律序列。这里把提取误差集中在查询端，便于解释实验结果。


In [ ]:
def extract_contour(y, sr=SR):
    # pYIN 提 f0，转 MIDI 音高，只保留有声帧，减中位数做移调归一
    f0, voiced_flag, _ = librosa.pyin(y, fmin=PYIN_FMIN, fmax=PYIN_FMAX, sr=sr, hop_length=PYIN_HOP)
    voiced = voiced_flag & ~np.isnan(f0)
    midi = 69.0 + 12.0 * np.log2(f0[voiced] / 440.0)
    return midi - np.median(midi)


def notes_to_contour(notes, frame_rate=FRAME_RATE):
    # 库端捷径：MIDI 音符按帧率直接采样成轮廓，同样减中位数
    t = np.arange(0.0, notes[-1].end, 1.0 / frame_rate)
    pitch = np.full(t.shape, np.nan)
    for n in notes:
        pitch[(t >= n.start) & (t < n.end)] = n.pitch
    pitch = pitch[~np.isnan(pitch)]
    return pitch - np.median(pitch)


# 渲染一条《茉莉花》合成哼唱：移高 5 半音，带默认扰动
moli_notes = core_songs["茉莉花"]
hum = render_humming(
    transpose_notes(moli_notes, QUERY_TRANSPOSE),
    HummingParams(sr=SR, detune_cents=DEFAULT_DETUNE, tempo_wander=DEFAULT_WANDER, seed=101),
)
contour_query = extract_contour(hum.wav)
contour_moli = notes_to_contour(moli_notes)
print(f"哼唱时长 {len(hum.wav) / SR:.1f} 秒,提取轮廓 {len(contour_query)} 帧;库内《茉莉花》轮廓 {len(contour_moli)} 帧")

truth_times = hum.times[::PYIN_HOP]
truth_pitch = hum.pitch_track[::PYIN_HOP] - np.nanmedian(hum.pitch_track)
fig, axes = plt.subplots(2, 1, figsize=(9, 4.6), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
axes[0].plot(np.arange(len(hum.wav)) / SR, hum.wav, color=LINE_GRAYS[2], linewidth=0.6)
axes[0].set_ylabel("振幅")
axes[1].plot(truth_times, truth_pitch, color=LINE_GRAYS[3], linewidth=1.0, label="渲染真值")
axes[1].plot(
    np.arange(len(contour_query)) / FRAME_RATE, contour_query, color=LINE_GRAYS[0], linewidth=1.2, label="pYIN 提取"
)
axes[1].set_xlabel("时间（秒）")
axes[1].set_ylabel("相对音高（半音）")
axes[1].legend(fontsize=8)
finish_figure(fig, OUTPUT_FIGURES / "09_2_pitch_contour.png")
plt.show()


## 2. 只记走向的 Parsons 码

1975 年出版的《The Directory of Tunes and Musical Themes》用简化的旋律轮廓帮助读者凭记忆查找曲名。相邻音符上行记 U、下行记 D、重复记 R，首音只作参照。《茉莉花》前六个音 A、G、A、C、D、C 编码为 DUUUD。

该目录收录约一万五千条曲调主题。原书由读者把记得的旋律方向转写成符号后查目录，并不是直接处理哼唱录音的系统。本 Notebook 用编辑距离比较两串走向码，替换、插入和删除的单位代价均为 1。

Parsons 码不记录音程大小和音符时长，五度跳进与二度级进都会记为 U。连续音高轮廓保留这些信息，但也更直接地受到音高估计误差影响。

当前基线用等长滑窗比较查询与库曲。库内走向码短于查询时没有可用窗口，代码记为无穷；这只是本实验协议的约定，编辑距离本身仍可比较不等长序列。走向码还使用渲染真值的音符分割，没有计入自动音符切分误差。


In [ ]:
def parsons_code(pitches):
    # 音符级 MIDI 音高(四舍五入到半音)转 U/D/R 走向串
    rounded = [int(round(p)) for p in pitches]
    signs = []
    for a, b in zip(rounded, rounded[1:]):
        signs.append("U" if b > a else "D" if b < a else "R")
    return "".join(signs)


def edit_distance(a, b):
    # Levenshtein 距离，单行滚动数组
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]


def parsons_best_window(query_code, lib_code):
    # 查询串对库串做等长滑窗，取最小编辑距离(哼唱常只哼一段)
    n = len(query_code)
    if n == 0 or len(lib_code) < n:
        return float("inf")
    return min(edit_distance(query_code, lib_code[k : k + n]) for k in range(len(lib_code) - n + 1))


# 查询取《茉莉花》前 20 秒的合成哼唱，走向码来自渲染后的真值音符
hum_parsons = render_humming(
    crop_notes(transpose_notes(moli_notes, QUERY_TRANSPOSE), 0.0, 20.0),
    HummingParams(sr=SR, detune_cents=DEFAULT_DETUNE, tempo_wander=DEFAULT_WANDER, seed=303),
)
query_code = parsons_code([n.pitch for n in hum_parsons.notes])
lib_codes = {name: parsons_code([n.pitch for n in notes]) for name, notes in core_songs.items()}
parsons_rows = sorted(
    ((name, parsons_best_window(query_code, code_str)) for name, code_str in lib_codes.items()),
    key=lambda r: r[1],
)
print(f"查询走向码:{query_code[:40]}...(共 {len(query_code)} 符)")
parsons_df = pd.DataFrame(parsons_rows, columns=["旋律", "Parsons 距离"])
parsons_df["Parsons 距离"] = [int(d) if np.isfinite(d) else "∞" for d in parsons_df["Parsons 距离"]]
print(parsons_df.to_string(index=False))


## 3. 动态时间规整

Parsons 码先把旋律压缩为走向符号；DTW 则直接对齐连续数值序列。给定查询轮廓 x 和库内轮廓 y，局部代价定义为 |x_i - y_j|，即两帧相差的半音数。

路径从两条序列的起点走到终点，每步允许查询前进一帧、库曲前进一帧，或两边同时前进。水平和垂直步让一侧的相邻多帧对应到另一侧同一帧，可表示局部时长差异或重复。

到达 (i, j) 的最小累计代价等于局部代价加上三个前驱中的最小值。填完整张矩阵后，从右下角沿最小前驱回溯得到对齐路径。

Sakoe-Chiba 带把递推限制在中心线附近，以减少计算并抑制过度规整。本实现按两条序列的长度比例设置中心线，并额外保证窗宽不小于长度差；这是代码采用的放宽，不是经典约束带的必要定义。

序列长度同量级且窗宽 w 远小于长度时，计算量可从 O(nm) 降到约 O(nw)。下面的手写实现包含递推与回溯，并与 `librosa.sequence.dtw` 对照。图中的深色区域表示局部代价低，浅色线表示最优路径。


In [ ]:
def dtw_align(x, y, window=None, subseq=False):
    # DTW：局部代价 |xi-yj|，三种步型，Sakoe-Chiba 带状窗
    # subseq=True 时第一行置零（起点可落在库轴任意位置，不计代价），终点在最后一行取最小
    n, m = len(x), len(y)
    if window is None:
        window = max(n, m)
    window = max(window, abs(n - m))  # 窗至少容纳两条序列的长度差
    D = np.full((n + 1, m + 1), np.inf)
    D[0, :] = 0.0 if subseq else np.inf
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        center = i * m / n
        j0 = max(1, int(np.floor(center - window)))
        j1 = min(m, int(np.ceil(center + window)))
        for j in range(j0, j1 + 1):
            D[i, j] = abs(x[i - 1] - y[j - 1]) + min(D[i - 1, j], D[i, j - 1], D[i - 1, j - 1])
    end_j = int(np.argmin(D[n, 1:])) + 1 if subseq else m
    path = []
    i, j = n, end_j
    while i > 0:
        path.append((i - 1, j - 1))
        prev = [(D[i - 1, j - 1], i - 1, j - 1), (D[i - 1, j], i - 1, j), (D[i, j - 1], i, j - 1)]
        _, i, j = min(prev, key=lambda s: s[0])
    path.reverse()
    return D[n, end_j], path, D


# 等长场景：完整《茉莉花》哼唱 对 库内《茉莉花》轮廓
cost_full, path_full, D_full = dtw_align(contour_query, contour_moli)
X = contour_query[np.newaxis, :]
D_lib, wp_lib = librosa.sequence.dtw(X=X, Y=contour_moli[np.newaxis, :])
print(f"手写 DTW 累计代价 {cost_full:.2f}，路径 {len(path_full)} 步，平均每步 {cost_full / len(path_full):.3f} 半音")
print(f"librosa 对照 {float(D_lib[-1, -1]):.2f}（应一致）")

# 代价矩阵用局部代价画（低代价是深色谷），叠加对齐路径
local_cost = np.abs(contour_query[:, None] - contour_moli[None, :])
fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.imshow(
    local_cost.T, origin="lower", aspect="auto", cmap="gray", vmin=0, vmax=12,
    extent=[0, len(contour_query) / FRAME_RATE, 0, len(contour_moli) / FRAME_RATE],
)
ax.plot(
    [p[0] / FRAME_RATE for p in path_full], [p[1] / FRAME_RATE for p in path_full],
    color="0.55", linewidth=1.4,
)
ax.set_xlabel("查询时间（秒）")
ax.set_ylabel("库内旋律时间（秒）")
finish_figure(fig, OUTPUT_FIGURES / "09_2_dtw_path.png")
plt.show()


## 4. 起点自由的子序列 DTW

全局 DTW 要求两条序列从开头到结尾完整对齐，而实际哼唱通常只覆盖一段。子序列 DTW 把累积矩阵第一行全部置零，使查询可从库曲任意位置开始；终点改为最后一行的最小值，使查询可在任意位置结束。

目标因此从比较两条完整序列，变为在长序列中寻找与查询最匹配的片段。回溯路径在库曲时间轴上覆盖的区间，即估计位置。

《苏武牧羊》查询实际从 13.8 秒附近开始，定位结果为 13.7–32.0 秒。《茉莉花》查询实际从 12.6 秒开始，却定位到 5.5–19.5 秒的相似乐句。换用无扰动的干净轮廓后落点不变，支持“旋律重复与当前局部代价共同造成歧义”的解释。


In [ ]:
# 从第 30% 处哼 20 秒，子序列 DTW 要在完整库旋律里找回这一段
suwu_notes = core_songs["苏武牧羊"]
contour_suwu = notes_to_contour(suwu_notes)
seg_notes = crop_notes(
    transpose_notes(suwu_notes, QUERY_TRANSPOSE),
    start=0.3 * suwu_notes[-1].end,
    end=0.3 * suwu_notes[-1].end + 20.0,
)
hum_seg = render_humming(
    seg_notes, HummingParams(sr=SR, detune_cents=DEFAULT_DETUNE, tempo_wander=DEFAULT_WANDER, seed=202)
)
contour_seg = extract_contour(hum_seg.wav)
cost_sub, path_sub, _ = dtw_align(contour_seg, contour_suwu, subseq=True)
D_lib_sub, wp_lib_sub = librosa.sequence.dtw(X=contour_seg[np.newaxis, :], Y=contour_suwu[np.newaxis, :], subseq=True)
seg_start = path_sub[0][1] / FRAME_RATE
seg_end = path_sub[-1][1] / FRAME_RATE
print(f"《苏武牧羊》片段实际取自 {0.3 * suwu_notes[-1].end:.1f} 秒处;子序列 DTW 定位到 {seg_start:.1f}–{seg_end:.1f} 秒")
print(f"手写累计代价 {cost_sub:.2f},librosa 对照 {float(D_lib_sub[-1].min()):.2f}(应一致)")

# 同一协议用在《茉莉花》中段：重复乐句把对齐引到前半首的相似段落
moli_notes = core_songs["茉莉花"]
contour_moli_full = notes_to_contour(moli_notes)
seg_moli = crop_notes(
    transpose_notes(moli_notes, QUERY_TRANSPOSE),
    start=0.3 * moli_notes[-1].end,
    end=0.3 * moli_notes[-1].end + 20.0,
)
hum_moli = render_humming(
    seg_moli, HummingParams(sr=SR, detune_cents=DEFAULT_DETUNE, tempo_wander=DEFAULT_WANDER, seed=202)
)
contour_moli_q = extract_contour(hum_moli.wav)
cost_moli, path_moli, _ = dtw_align(contour_moli_q, contour_moli_full, subseq=True)
moli_start = path_moli[0][1] / FRAME_RATE
moli_end = path_moli[-1][1] / FRAME_RATE
print(f"《茉莉花》片段实际取自 {0.3 * moli_notes[-1].end:.1f} 秒处;定位到 {moli_start:.1f}–{moli_end:.1f} 秒(前半首的相似乐句)")

# 对照：绕过合成与音频提取，真值轮廓直接当查询，落点不变则失配来自旋律自身的重复
contour_moli_clean = notes_to_contour(
    crop_notes(moli_notes, start=0.3 * moli_notes[-1].end, end=0.3 * moli_notes[-1].end + 20.0)
)
cost_clean, path_clean, _ = dtw_align(contour_moli_clean, contour_moli_full, subseq=True)
clean_start = path_clean[0][1] / FRAME_RATE
clean_end = path_clean[-1][1] / FRAME_RATE
print(f"干净轮廓对照:定位到 {clean_start:.1f}–{clean_end:.1f} 秒,落点不变")

# 左右对照：灰度为局部代价矩阵，灰线为对齐路径
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
panels = [
    (axes[0], contour_seg, contour_suwu, path_sub, "《苏武牧羊》：定位成功"),
    (axes[1], contour_moli_q, contour_moli_full, path_moli, "《茉莉花》：落到相似乐句"),
]
for ax, cq, cl, path, title in panels:
    ax.imshow(
        np.abs(cq[:, None] - cl[None, :]).T, origin="lower", aspect="auto", cmap="gray", vmin=0, vmax=12,
        extent=[0, len(cq) / FRAME_RATE, 0, len(cl) / FRAME_RATE],
    )
    ax.plot(
        [p[0] / FRAME_RATE for p in path], [p[1] / FRAME_RATE for p in path],
        color="0.55", linewidth=1.4,
    )
    ax.set_xlabel("查询时间（秒）")
    ax.set_ylabel("库内旋律时间（秒）")
    ax.set_title(title)
finish_figure(fig, OUTPUT_FIGURES / "09_2_subsequence_path.png")
plt.show()


## 5. 音高偏移与节奏漂移的鲁棒性扫描

合成哼唱便于控制变量。实验固定随机种子，每次只改变一种扰动：逐音音高偏移的标准差，或平滑节奏漂移的强度。查询取八首旋律各自的前 45 秒，指标为平均倒数排名（MRR）。

不加音高偏移时，DTW 与 Parsons 在八首小库上均得到 MRR 1.0。110 音分时，Parsons 有一条查询排到第二，MRR 为 0.938；DTW 只在 20 音分一档有一条排到第三，MRR 为 0.917，其余档位为 1.0。

结果没有随扰动强度单调下降。这说明八条查询、单次随机样本的扫描容易受具体旋律与扰动实例影响，证据不足以估计稳定的鲁棒性阈值，也不足以判断某种表示在整个范围内更优。

Parsons 直接使用生成器保存的真值音符顺序，并把音高四舍五入到半音后记录方向；DTW 使用 pYIN 提取的帧级轮廓。两条流程除表示和匹配器外，输入条件也不同。节奏轴只报告 DTW，因为当前 Parsons 对照不使用音符时长；真实录音中的音符切分误差不在该对照范围内。


In [ ]:
def dtw_rank_table(query_contour, lib_contours):
    # 一条查询对整个库：子序列 DTW(查询对齐进库曲)，按每步平均代价排序
    rows = []
    for name, contour in lib_contours.items():
        D, wp = librosa.sequence.dtw(C=np.abs(query_contour[:, None] - contour[None, :]), subseq=True)
        j = int(np.argmin(D[-1]))
        rows.append((name, float(D[-1, j]) / len(wp)))
    return sorted(rows, key=lambda r: r[1])


def mrr_of(rank_tables, truth_names):
    # rank_tables: 每条查询的排序表；truth_names: 对应的真实答案
    total = 0.0
    for table, truth in zip(rank_tables, truth_names):
        rank = next(i for i, (name, _) in enumerate(table, 1) if name == truth)
        total += 1.0 / rank
    return total / len(rank_tables)


def parsons_rank_table(query_notes, lib_codes):
    code_q = parsons_code([n.pitch for n in query_notes])
    rows = [(name, parsons_best_window(code_q, code_lib)) for name, code_lib in lib_codes.items()]
    return sorted(rows, key=lambda r: r[1])


core_contours = {name: notes_to_contour(notes) for name, notes in core_songs.items()}


def run_queries(detune, wander, seed0=1000):
    # 八首各一条 45 秒哼唱片段，返回 DTW 与 Parsons 两种排序表
    dtw_tables, parsons_tables, truths = [], [], []
    for k, (name, notes) in enumerate(core_songs.items()):
        seg = crop_notes(transpose_notes(notes, QUERY_TRANSPOSE), 0.0, QUERY_SEC)
        hum_k = render_humming(seg, HummingParams(sr=SR, detune_cents=detune, tempo_wander=wander, seed=seed0 + k))
        dtw_tables.append(dtw_rank_table(extract_contour(hum_k.wav), core_contours))
        parsons_tables.append(parsons_rank_table(hum_k.notes, lib_codes))
        truths.append(name)
    return dtw_tables, parsons_tables, truths


detune_grid = [0, 20, 40, 70, 110]
wander_grid = [0.0, 0.03, 0.06, 0.12]
detune_rows, wander_rows = [], []
for detune in detune_grid:
    dtw_tables, parsons_tables, truths = run_queries(detune, DEFAULT_WANDER)
    detune_rows.append(
        {"音高偏移标准差(音分)": detune, "DTW MRR": round(mrr_of(dtw_tables, truths), 3),
         "Parsons MRR": round(mrr_of(parsons_tables, truths), 3)}
    )
for wander in wander_grid:
    dtw_tables, _, truths = run_queries(DEFAULT_DETUNE, wander)
    wander_rows.append({"节奏漂移(秒)": wander, "DTW MRR": round(mrr_of(dtw_tables, truths), 3)})
detune_df = pd.DataFrame(detune_rows)
wander_df = pd.DataFrame(wander_rows)
print(detune_df.to_string(index=False))
print(wander_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(detune_df["音高偏移标准差(音分)"], detune_df["DTW MRR"], marker="o", color=LINE_GRAYS[0], label="DTW")
axes[0].plot(detune_df["音高偏移标准差(音分)"], detune_df["Parsons MRR"], marker="s", color=LINE_GRAYS[3], label="Parsons")
axes[0].set_xlabel("音高偏移标准差（音分）")
axes[0].set_ylabel("MRR")
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=8)
axes[1].plot(wander_df["节奏漂移(秒)"], wander_df["DTW MRR"], marker="o", color=LINE_GRAYS[0])
axes[1].set_xlabel("节奏漂移强度（秒）")
axes[1].set_ylabel("DTW MRR")
axes[1].set_ylim(0, 1.05)
finish_figure(fig, OUTPUT_FIGURES / "09_2_robustness.png")
plt.show()


## 6. 曲库规模的扩张

本节把曲库从 8 首逐步扩到 422 首。规模层来自 Essen 民歌集中国子集；按本项目取数时使用的仓库版本，四个目录 han、shanxi、natmin、xinhua 分别含 1223、802、206、10 个 `.krn` 文件，共 2241 个。

这里保留仓库目录名，不把它们自行解释为更具体的地域或族群标签。规模层从 han 和 shanxi 抽取 120 与 78 首，并保留 natmin 与 xinhua 的全部文件，共 414 首；再与八首核心旋律合并。

同一批八条查询在 8、50、100、200、422 首曲库上的 MRR 分别为 1.000、1.000、0.938、0.875、0.838，Top-1 命中由 8/8 降至 6/8。新增候选可能出现更相似的轮廓，从而改变正确作品的名次。

串行逐曲 DTW 的总耗时随候选数近似线性增长。下方代码按本次平均单曲比较耗时，外推 2241 首和十万首曲库的查询时间。结果会随硬件、序列长度、系统负载和实现变化，只应作为量级估计。

FastDTW 用多分辨率方法近似对齐，速度与误差取决于搜索半径和数据。另一类方案先用旋律片段索引、符号 n-gram 或嵌入近邻筛选候选，再对少数候选做精确 DTW。两类方案都需在目标数据上验证。


In [ ]:
# Essen 414 首：按 manifest 读取，与核心库合成 422 首实验库
essen_manifest = pd.read_csv(ESSEN_DIR / "manifest.csv")
essen_contours = {}
t0 = time.perf_counter()
for _, row in essen_manifest.iterrows():
    notes_k = load_midi_notes(ESSEN_DIR / row["midi_path"])
    if len(notes_k) < 8:
        continue
    essen_contours[f"{row['subcorpus']}/{Path(row['midi_path']).stem}"] = notes_to_contour(notes_k)
print(f"Essen 轮廓 {len(essen_contours)} 条,构建耗时 {time.perf_counter() - t0:.1f} 秒")

full_contours = {**core_contours, **essen_contours}
rng = np.random.default_rng(42)
essen_names = sorted(essen_contours)
order = rng.permutation(len(essen_names))
tier_sizes = [8, 50, 100, 200, len(full_contours)]
tier_libs = {}
for k in tier_sizes:
    extra = sorted(essen_names[i] for i in order[: max(0, k - 8)])
    tier_libs[k] = {**core_contours, **{n: essen_contours[n] for n in extra}}

# 八条 45 秒合成哼唱(默认扰动)，对全库算一遍距离，各档规模从同一距离矩阵取排名
query_contours, query_truths = [], []
for k, (name, notes) in enumerate(core_songs.items()):
    seg = crop_notes(transpose_notes(notes, QUERY_TRANSPOSE), 0.0, QUERY_SEC)
    hum_k = render_humming(
        seg, HummingParams(sr=SR, detune_cents=DEFAULT_DETUNE, tempo_wander=DEFAULT_WANDER, seed=3000 + k)
    )
    query_contours.append(extract_contour(hum_k.wav))
    query_truths.append(name)

t0 = time.perf_counter()
dist_matrix = np.zeros((len(query_contours), len(full_contours)))
lib_names = list(full_contours)
# 传 C 矩阵固定语义：查询(行)始终对齐进库曲(列)，与两边长度关系无关
for qi, contour_q in enumerate(query_contours):
    for li, name in enumerate(lib_names):
        D, wp = librosa.sequence.dtw(C=np.abs(contour_q[:, None] - full_contours[name][None, :]), subseq=True)
        dist_matrix[qi, li] = float(D[-1].min()) / len(wp)
scan_all = time.perf_counter() - t0
print(f"全库扫描 {len(query_contours)} × {len(lib_names)} 对,耗时 {scan_all:.1f} 秒")

scale_rows = []
for k in tier_sizes:
    keep = [lib_names.index(n) for n in tier_libs[k]]
    sub = dist_matrix[:, keep]
    names_k = [lib_names[i] for i in keep]
    ranks = []
    top1 = 0
    for qi, truth in enumerate(query_truths):
        sorted_idx = np.argsort(sub[qi])
        rank = next(r for r, idx in enumerate(sorted_idx, 1) if names_k[idx] == truth)
        ranks.append(rank)
        top1 += rank == 1
    scale_rows.append(
        {"库规模": k, "Top-1 命中": f"{top1}/{len(query_truths)}", "MRR": round(float(np.mean([1 / r for r in ranks])), 3)}
    )
scale_df = pd.DataFrame(scale_rows)
print(scale_df.to_string(index=False))

# 单查询耗时随规模：对每档子集用同一条查询实测
contour_q0 = query_contours[0]
time_rows = []
for k in tier_sizes:
    t0 = time.perf_counter()
    dtw_rank_table(contour_q0, tier_libs[k])
    per_query = time.perf_counter() - t0
    time_rows.append({"库规模": k, "单查询耗时(秒)": round(per_query, 2)})
time_df = pd.DataFrame(time_rows)
print(time_df.to_string(index=False))
est_full = time_df.iloc[-1]["单查询耗时(秒)"] / len(full_contours)
print(f"按线性外推:2241 首约 {est_full * 2241:.0f} 秒,10 万首约 {est_full * 100000 / 60:.0f} 分钟")

# 最大规模下一条查询的 Top-5：真实答案排在首位，紧随其后的是最相似的库内近邻
top_table = dtw_rank_table(contour_q0, full_contours)[:5]
print(f"查询《{query_truths[0]}》在全库的 Top-5:")
print(pd.DataFrame(top_table, columns=["旋律", "每步平均代价"]).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(scale_df["库规模"], scale_df["MRR"], marker="o", color=LINE_GRAYS[0])
axes[0].set_xlabel("库规模（首）")
axes[0].set_ylabel("MRR")
axes[0].set_ylim(0, 1.05)
axes[0].set_xscale("log")
axes[1].plot(time_df["库规模"], time_df["单查询耗时(秒)"], marker="s", color=LINE_GRAYS[0])
axes[1].set_xlabel("库规模（首）")
axes[1].set_ylabel("单查询耗时（秒）")
axes[1].set_xscale("log")
finish_figure(fig, OUTPUT_FIGURES / "09_2_scale.png")
plt.show()


## 7. 真实哼唱与公开数据

合成实验只包含预设的四类变化，而且示例中的基频估计误差较小。真实哼唱还可能包含装饰音、更加自由的滑音、气声、断句和音符遗漏，使内容偏差与基频提取误差同时出现。

如需加入真实录音，可把 WAV 文件放入 `CODE/datasets/audio_author/chapter_09_author/humming` 后运行下方单元。目录为空时只报告状态。一两条录音可检查流程能否处理真实文件，但不足以支持性能结论。

腾讯音乐天琴实验室发布的 Lyra-QBH 含 97 位用户（38 男、59 女）通过手机无伴奏录制的 1005 个片段，覆盖 100 首歌曲。官方页面给出的有效时长为 9–10 秒，平均 9.98 秒；音频为 8 kHz、16 位、单声道 WAV，并附 100 个 MIDI 文件。

数据采用 CC BY-NC 4.0，获取前还需按页面要求提交申请。若用于扩展本实验，还要核对曲目对应、切分方式和评价协议，不宜因两边都提供 MIDI 就直接并入当前民歌库。

Ghias 等人在 1995 年提出了从哼唱查询音乐数据库的方法，Typke 等人在 2005 年综述了早期旋律检索系统。不同工作对表示粒度、查询误差和匹配算法采取了不同选择。


In [ ]:
# 真实录音对照：哼唱文件按 author_audio_materials_spec 流程入库后，重跑本单元即可
author_wavs = sorted(AUTHOR_HUM_DIR.glob("*.wav")) if AUTHOR_HUM_DIR.exists() else []
if not author_wavs:
    print(f"哼唱文件尚未入库（期待位置 {rel(AUTHOR_HUM_DIR)}），本单元仅报告状态")
for wav_path in author_wavs:
    y_real, _ = load_audio(wav_path, sr=SR)
    table = dtw_rank_table(extract_contour(y_real), core_contours)[:3]
    print(f"{wav_path.stem} 的 Top-3: {[(name, round(c, 3)) for name, c in table]}")

detune_df.to_csv(OUTPUT_TABLES / "09_2_robustness_detune.csv", index=False)
wander_df.to_csv(OUTPUT_TABLES / "09_2_robustness_wander.csv", index=False)
scale_df.to_csv(OUTPUT_TABLES / "09_2_scale_mrr.csv", index=False)
time_df.to_csv(OUTPUT_TABLES / "09_2_scale_time.csv", index=False)
pd.DataFrame(top_table, columns=["旋律", "每步平均代价"]).to_csv(OUTPUT_TABLES / "09_2_full_library_top5.csv", index=False)
print("表格已写入", rel(OUTPUT_TABLES))
print("图已写入", rel(OUTPUT_FIGURES))
